# Kubeflow Trainer × OpenTelemetry — Pain Point Showcase (KIND / CPU)

This notebook demonstrates **two real developer pain points** in distributed training
and shows exactly how OTel traces surface them — without any GPU or heavy computation.
All jobs run on a KIND cluster with CPU-only workers.

## Pain points demonstrated

| # | Pain point | Without OTel | With OTel |
|---|---|---|---|
| 1 | **Straggler worker** — one rank is slow, every other rank waits at `dist.barrier()` | `kubectl logs` from each pod separately, manual timestamp diffing | Single waterfall: `training.worker [rank 1]` visually 3× wider |
| 2 | **Mid-training failure** — job crashes at epoch 2 on rank 0 | Scan pod logs for stack trace, no context on what loss was at crash | Red ERROR span in Jaeger, click it: epoch number + loss at crash + exception message, all in one place |

## Trace flow (both scenarios)
```
notebook (SDK)
  └─ train job                     ← SDK span (TrainerClient.train)
       └─ k8s create train job     ← backend span (K8sBackend)
            ├─ training.worker [rank 0]   ← worker pod span (CONSUMER)
            │    ├─ training.epoch [1]    ← per-epoch span (INTERNAL)
            │    ├─ training.epoch [2]    ← ERROR span in scenario 2
            │    └─ training.epoch [3]
            └─ training.worker [rank 1]   ← straggler in scenario 1
                 ├─ training.epoch [1]    ← 3× longer duration
                 ├─ training.epoch [2]
                 └─ training.epoch [3]
```

In [24]:
import kubeflow
print(kubeflow.__file__)

/Users/sujalshah/Desktop/code/opensource/kubeflow/sdk/kubeflow/__init__.py


## OTel Setup
Build a TracerProvider that exports to Jaeger via OTLP gRPC.
Swap `backend="console"` if you want spans printed to stdout instead.

In [25]:
def build_tracer_provider(
    service_name: str = "kubeflow-trainer-example",
    backend: str = "jaeger",   # "jaeger" | "otlp-collector" | "console"
    endpoint: str = "http://localhost:4317",
    headers: dict = None,
):
    """
    Build and return a TracerProvider configured for the chosen backend.
    """
    from opentelemetry.propagate import set_global_textmap
    from opentelemetry.sdk.resources import Resource
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import BatchSpanProcessor
    from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator

    resource = Resource.create({
        "service.name": service_name,
        "service.version": "0.1.0",
    })

    provider = TracerProvider(resource=resource)

    if backend in ("jaeger", "otlp-collector"):
        from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
        exporter = OTLPSpanExporter(
            endpoint=endpoint,
            headers=headers or {},
            insecure=True,
        )
    elif backend == "console":
        from opentelemetry.sdk.trace.export import ConsoleSpanExporter
        exporter = ConsoleSpanExporter()
    else:
        raise ValueError(f"Unknown backend '{backend}'.")

    provider.add_span_processor(BatchSpanProcessor(exporter))

    # W3C TraceContext — sets traceparent / tracestate headers.
    set_global_textmap(TraceContextTextMapPropagator())

    return provider

---
## Scenario 1 — Straggler Worker

### The problem today (without OTel)
Rank 1 is slow — maybe it got scheduled on a noisy node, maybe NUMA locality is off.
Every other rank sits blocked at `dist.barrier()` waiting for it.
The job takes 3× longer than it should.

To find the culprit a developer today has to:
1. `kubectl logs <job>-node-0` — scan timestamps manually
2. `kubectl logs <job>-node-1` — scan timestamps manually  
3. Diff the two outputs by eye and guess which rank is behind

There is no single view. There is no structured duration per rank per epoch.

### What the trace shows
- `training.worker [rank 1]` span is visually ~3× wider than `training.worker [rank 0]`
- Each `training.epoch` span carries `training.epoch_duration_ms` — queryable in Jaeger
- The parent `k8s create train job` span duration reflects the real wall-clock cost including the straggler wait
- **Zero manual log diffing. One waterfall view.**

> In this simulation rank 1 sleeps 0.3s extra per epoch vs rank 0's 0.1s.
> No GPU or heavy compute — identical behaviour to a real slow node from the trace perspective.

In [26]:
def train_fn_straggler():
    """
    Scenario 1: Straggler worker.
    Rank 1 sleeps 3x longer per epoch than rank 0, simulating a slow node.
    Every rank waits at dist.barrier() — so rank 0's total time is bottlenecked by rank 1.
    The OTel trace makes the gap instantly visible; kubectl logs do not.
    """
    import os
    import time
    import torch
    import torch.distributed as dist
    import subprocess, sys

    try:
        import opentelemetry
    except ImportError:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--break-system-packages",
            "opentelemetry-sdk",
            "opentelemetry-exporter-otlp-proto-grpc",
        ])

    from opentelemetry import trace
    from opentelemetry.propagate import extract, set_global_textmap
    from opentelemetry.sdk.resources import Resource
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import BatchSpanProcessor
    from opentelemetry.trace import SpanKind
    from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator

    set_global_textmap(TraceContextTextMapPropagator())

    carrier = {
        "traceparent": os.environ.get("TRACEPARENT", ""),
        "tracestate":  os.environ.get("TRACESTATE",  ""),
    }
    parent_ctx = extract(carrier)

    otlp_endpoint = os.environ.get(
        "OTEL_EXPORTER_OTLP_ENDPOINT",
        "http://jaeger.default.svc.cluster.local:4317",
    )
    rank = int(os.environ.get("RANK", "0"))

    resource = Resource.create({
        "service.name":    "kubeflow-trainer-worker",
        "service.version": "0.1.0",
        "k8s.pod.name":    os.environ.get("POD_NAME", f"worker-{rank}"),
        "training.rank":   rank,
    })
    provider = TracerProvider(resource=resource)

    try:
        from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
        exporter = OTLPSpanExporter(endpoint=otlp_endpoint, insecure=True)
    except ImportError:
        from opentelemetry.sdk.trace.export import ConsoleSpanExporter
        exporter = ConsoleSpanExporter()

    provider.add_span_processor(BatchSpanProcessor(exporter))
    tracer = provider.get_tracer("kubeflow.trainer.worker")

    # Rank 1 is the straggler: 0.3s per epoch vs rank 0's 0.1s.
    # Both ranks call dist.barrier() after each epoch — rank 0 blocks waiting for rank 1.
    epoch_sleep = 0.3 if rank == 1 else 0.1

    with tracer.start_as_current_span(
        "training.worker",
        kind=SpanKind.CONSUMER,
        context=parent_ctx,
    ) as worker_span:
        worker_span.set_attribute("training.rank",        rank)
        worker_span.set_attribute("training.backend",     "gloo")
        worker_span.set_attribute("training.num_epochs",  3)
        worker_span.set_attribute("training.is_straggler", rank == 1)  # tag the slow rank

        dist.init_process_group(backend="gloo")
        world_size = dist.get_world_size()
        worker_span.set_attribute("training.world_size", world_size)
        print(f"[rank {rank}/{world_size}] init done — epoch_sleep={epoch_sleep}s")

        for epoch in range(1, 4):
            epoch_start = time.time()

            with tracer.start_as_current_span(
                "training.epoch",
                kind=SpanKind.INTERNAL,
            ) as epoch_span:
                epoch_span.set_attribute("training.epoch", epoch)
                epoch_span.set_attribute("training.rank",  rank)

                # Simulate work — rank 1 takes 3x longer
                t    = torch.randn(64, 64)
                loss = (t @ t.T).mean().item()
                time.sleep(epoch_sleep)

                duration_ms = round((time.time() - epoch_start) * 1000, 2)
                epoch_span.set_attribute("training.loss",             round(loss, 6))
                epoch_span.set_attribute("training.epoch_duration_ms", duration_ms)

            # All ranks sync here — rank 0 blocks until rank 1 finishes
            dist.barrier()
            print(f"[rank {rank}] epoch {epoch}/3  loss={loss:.4f}  duration={duration_ms}ms")

        dist.barrier()
        if rank == 0:
            print("Training complete")

        dist.destroy_process_group()

    provider.force_flush()

### Submit — Scenario 1 (Straggler)

**What to look for in Jaeger after this runs:**
- Search for service `kubeflow-trainer-worker`
- Open the trace — two `training.worker` spans side by side
- `training.worker [rank 1]` is ~3× wider than `training.worker [rank 0]`
- Click any `training.epoch` span on rank 1 → see `training.epoch_duration_ms ≈ 300ms`
- Click any `training.epoch` span on rank 0 → see `training.epoch_duration_ms ≈ 100ms`
- The `training.is_straggler = true` attribute on rank 1's worker span confirms the slow node

**This is what takes 30 minutes with kubectl logs. With the trace it takes 10 seconds.**

In [27]:
from kubeflow.trainer import TrainerClient, CustomTrainer
from opentelemetry import trace

provider = build_tracer_provider(
    service_name="kubeflow-trainer-example",
    backend="jaeger",
    endpoint="http://localhost:4317",
)
client = TrainerClient(tracer_provider=provider)

torch_runtime = None
for runtime in client.list_runtimes():
    if runtime.name == "torch-distributed":
        torch_runtime = runtime
        break

assert torch_runtime is not None, "torch-distributed runtime not found"

job_name_straggler = client.train(
    trainer=CustomTrainer(
        func=train_fn_straggler,
        num_nodes=2,
        resources_per_node={"cpu": 1},
    ),
    runtime=torch_runtime,
)
print(f"Submitted straggler job: {job_name_straggler}")

DEBUG carrier: {'traceparent': '00-aa764028a309a7c0b601bc0f83f06e0d-c1a05b1fc18f9390-01'}
DEBUG span valid: True
DEBUG span id: 0xc1a05b1fc18f9390
DEBUG [IoK8sApiCoreV1EnvVar(name='TRACEPARENT', value='00-aa764028a309a7c0b601bc0f83f06e0d-c1a05b1fc18f9390-01', value_from=None)]

Submitted straggler job: w445b9eef68c


In [28]:
# Stream logs — notice rank 1 prints each epoch later than rank 0,
# but from logs alone you cannot see the duration gap or which rank caused it.
print("=== node-0 (rank 0) logs ===")
for line in client.get_job_logs(job_name_straggler, follow=True, step="node-0"):
    print(line, end="")

print("\n=== node-1 (rank 1) logs ===")
for line in client.get_job_logs(job_name_straggler, follow=False, step="node-1"):
    print(line, end="")

=== node-0 (rank 0) logs ===

=== node-1 (rank 1) logs ===


In [29]:
client.delete_job(job_name_straggler)

---
## Scenario 2 — Mid-Training Failure (Rank 0 crashes at epoch 2)

### The problem today (without OTel)
A distributed job crashes mid-way. The error could be OOM, a NaN loss, a network timeout,
or a bug in user code. Kubernetes marks the pod as `Error` or `OOMKilled`.

To diagnose, a developer today:
1. Runs `kubectl get pods` — sees one pod in `Error` state
2. Runs `kubectl logs <failed-pod>` — gets a wall of text with a stack trace somewhere inside
3. Has no idea what epoch failed, what the loss was at the moment of failure, or whether the other rank had already progressed further
4. Has to cross-reference both pod logs by timestamp to reconstruct the sequence of events

### What the trace shows
- `training.epoch [2]` span on rank 0 is marked **ERROR** (red in Jaeger)
- Clicking it shows: `training.epoch = 2`, `training.loss = <last value>`, `exception.message`, `exception.stacktrace`
- `training.worker [rank 0]` is also marked ERROR and shows total duration before crash
- `training.worker [rank 1]` completed successfully — visible in the same trace
- **The full failure context is in one span. No log scanning.**

> Rank 0 raises a `RuntimeError` at epoch 2 to simulate a training failure (NaN loss, assertion error, etc.).
> The exception is recorded on the span with `span.record_exception()` before re-raising.

In [30]:
def train_fn_failure():
    """
    Scenario 2: Mid-training failure.
    Rank 0 crashes at epoch 2 with a RuntimeError (simulating NaN loss / assertion failure).
    The OTel span captures the exception, the epoch number, and the last loss value.
    Without OTel, this context exists only as a stack trace buried in pod logs.
    """
    import os
    import time
    import torch
    import torch.distributed as dist
    import subprocess, sys

    try:
        import opentelemetry
    except ImportError:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--break-system-packages",
            "opentelemetry-sdk",
            "opentelemetry-exporter-otlp-proto-grpc",
        ])

    from opentelemetry import trace
    from opentelemetry.propagate import extract, set_global_textmap
    from opentelemetry.sdk.resources import Resource
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import BatchSpanProcessor
    from opentelemetry.trace import SpanKind, StatusCode
    from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator

    set_global_textmap(TraceContextTextMapPropagator())

    carrier = {
        "traceparent": os.environ.get("TRACEPARENT", ""),
        "tracestate":  os.environ.get("TRACESTATE",  ""),
    }
    parent_ctx = extract(carrier)

    otlp_endpoint = os.environ.get(
        "OTEL_EXPORTER_OTLP_ENDPOINT",
        "http://jaeger.default.svc.cluster.local:4317",
    )
    rank = int(os.environ.get("RANK", "0"))

    resource = Resource.create({
        "service.name":    "kubeflow-trainer-worker",
        "service.version": "0.1.0",
        "k8s.pod.name":    os.environ.get("POD_NAME", f"worker-{rank}"),
        "training.rank":   rank,
    })
    provider = TracerProvider(resource=resource)

    try:
        from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
        exporter = OTLPSpanExporter(endpoint=otlp_endpoint, insecure=True)
    except ImportError:
        from opentelemetry.sdk.trace.export import ConsoleSpanExporter
        exporter = ConsoleSpanExporter()

    provider.add_span_processor(BatchSpanProcessor(exporter))
    tracer = provider.get_tracer("kubeflow.trainer.worker")

    with tracer.start_as_current_span(
        "training.worker",
        kind=SpanKind.CONSUMER,
        context=parent_ctx,
    ) as worker_span:
        worker_span.set_attribute("training.rank",       rank)
        worker_span.set_attribute("training.backend",    "gloo")
        worker_span.set_attribute("training.num_epochs", 3)

        dist.init_process_group(backend="gloo")
        world_size = dist.get_world_size()
        worker_span.set_attribute("training.world_size", world_size)
        print(f"[rank {rank}/{world_size}] init done")

        try:
            for epoch in range(1, 4):
                with tracer.start_as_current_span(
                    "training.epoch",
                    kind=SpanKind.INTERNAL,
                ) as epoch_span:
                    epoch_span.set_attribute("training.epoch", epoch)
                    epoch_span.set_attribute("training.rank",  rank)

                    t    = torch.randn(64, 64)
                    loss = (t @ t.T).mean().item()
                    time.sleep(0.1)

                    epoch_span.set_attribute("training.loss", round(loss, 6))

                    # ── Rank 0 crashes at epoch 2 ──────────────────────────────────
                    # Simulates: NaN loss, assertion failure, OOM, or any runtime error.
                    # The exception is recorded ON THE SPAN before re-raising —
                    # so Jaeger shows epoch=2, loss=<last value>, and the full stack trace
                    # all in one place, without touching kubectl logs.
                    if rank == 0 and epoch == 2:
                        err = RuntimeError(
                            f"Loss diverged at epoch {epoch}: "
                            f"loss={loss:.4f} exceeded threshold. "
                            "Simulated training failure."
                        )
                        epoch_span.record_exception(err)
                        epoch_span.set_status(StatusCode.ERROR, description=str(err))
                        worker_span.set_status(StatusCode.ERROR, description=f"Worker failed at epoch {epoch}")
                        raise err

                dist.barrier()
                print(f"[rank {rank}] epoch {epoch}/3  loss={loss:.4f}")

        except Exception as exc:
            # Ensure spans are flushed before the process exits on failure
            print(f"[rank {rank}] FAILED: {exc}")
            provider.force_flush()
            raise

        dist.barrier()
        if rank == 0:
            print("Training complete")

        dist.destroy_process_group()

    provider.force_flush()

### Submit — Scenario 2 (Failure)

**What to look for in Jaeger after this runs:**
- Open the trace — look for the red ERROR spans
- `training.epoch [2]` on rank 0 is red → click it
  - `training.epoch = 2` ← exactly which epoch failed
  - `training.loss = <value>` ← what the loss was at the moment of crash  
  - `exception.message` ← the error message
  - `exception.stacktrace` ← full stack trace, structured, not buried in log lines
- `training.worker [rank 0]` is also red — total worker span marked as failed
- `training.worker [rank 1]` is green — it completed epoch 1 before rank 0 crashed at barrier
- **The sequence of events across both pods is visible in one view.**

Note: rank 1 will also fail/hang at `dist.barrier()` after epoch 1 because rank 0 died.
This is expected — it mirrors what happens in a real distributed failure.

In [31]:
from kubeflow.trainer import TrainerClient, CustomTrainer
from opentelemetry import trace


torch_runtime = None
for runtime in client.list_runtimes():
    if runtime.name == "torch-distributed":
        torch_runtime = runtime
        break

assert torch_runtime is not None, "torch-distributed runtime not found"

job_name_failure = client.train(
    trainer=CustomTrainer(
        func=train_fn_failure,
        num_nodes=2,
        resources_per_node={"cpu": 1},
    ),
    runtime=torch_runtime,
)
print(f"Submitted failure job: {job_name_failure}")

DEBUG carrier: {'traceparent': '00-35ba79c8faac20d6982e21b0b5455f82-df2c5875224f511b-01'}
DEBUG span valid: True
DEBUG span id: 0xdf2c5875224f511b
DEBUG [IoK8sApiCoreV1EnvVar(name='TRACEPARENT', value='00-35ba79c8faac20d6982e21b0b5455f82-df2c5875224f511b-01', value_from=None)]

Submitted failure job: z84bbda06ed4


In [32]:
# Stream logs — notice rank 0 prints an error, but the log alone tells you
# nothing about which epoch failed or what the loss was. The span has all of it.
print("=== node-0 (rank 0) logs — expect failure at epoch 2 ===")
for line in client.get_job_logs(job_name_failure, follow=True, step="node-0"):
    print(line, end="")

print("\n=== node-1 (rank 1) logs ===")
for line in client.get_job_logs(job_name_failure, follow=False, step="node-1"):
    print(line, end="")

=== node-0 (rank 0) logs — expect failure at epoch 2 ===
=== node-1 (rank 1) logs ===

In [15]:
client.delete_job(job_name_failure)

RuntimeError: Failed to delete TrainJob: default/v190affeb126